In [1]:
import ollama
import json
import pandas as pd

In [2]:
# Load QULAC dataset from local file
# This file contains queries, clarification questions, and metadata
with open("C:\Raina's Files\Semester IV - Thesis\Data\qulac.json") as f:
    data = json.load(f)

# convert to normal rows
df = pd.DataFrame(data)

print(df.head())

<>:3: SyntaxWarning: invalid escape sequence '\R'
<>:3: SyntaxWarning: invalid escape sequence '\R'
C:\Users\hardi\AppData\Local\Temp\ipykernel_5208\2560555819.py:3: SyntaxWarning: invalid escape sequence '\R'
  with open("C:\Raina's Files\Semester IV - Thesis\Data\qulac.json") as f:


   topic_id  facet_id topic_facet_id topic_facet_question_id  \
0         1         1            1-1                   1-1-1   
1         1         1            1-1                  1-1-10   
2         1         1            1-1                  1-1-11   
3         1         1            1-1                  1-1-12   
4         1         1            1-1                   1-1-2   

               topic topic_type facet_type  \
0  obama family tree    faceted        nav   
1  obama family tree    faceted        nav   
2  obama family tree    faceted        nav   
3  obama family tree    faceted        nav   
4  obama family tree    faceted        nav   

                                          topic_desc  \
0  Find information on President Barack Obama\'s ...   
1  Find information on President Barack Obama\'s ...   
2  Find information on President Barack Obama\'s ...   
3  Find information on President Barack Obama\'s ...   
4  Find information on President Barack Obama\'s ...   

 

In [3]:
# keep only id, query, question
df_small = df[['topic_id', 'topic', 'question']]

# remove duplicates
df_small = df_small.drop_duplicates(subset=['topic', 'question'])

# sample 30 rows
sample_df = df_small.sample(n=30, random_state=42)

# reset index for clean IDs
sample_df = sample_df.reset_index(drop=True)
sample_df['ID'] = sample_df.index + 1

# reorder columns
sample_df = sample_df[['ID', 'topic', 'question']]

print(sample_df.head(10))

   ID                       topic  \
0   1     dutchess county tourism   
1   2  ontario california airport   
2   3        civil right movement   
3   4                    flushing   
4   5    map of the united states   
5   6      von willebrand disease   
6   7  furniture for small spaces   
7   8  ontario california airport   
8   9      electronic skeet shoot   
9  10                       atari   

                                            question  
0  what tourist spots are you looking for in dutc...  
1  is there anywhere in california youd like to g...  
2       do you want to know who started the movement  
3              would you like to know about blushing  
4  do you want to see a map of the continental un...  
5  are you looking for specific information about...  
6          are you looking for dining room furniture  
7                                                     
8          are you looking for reviews related to it  
9        would you like to see a list of a

In [4]:
# drop empty question
sample_df = sample_df[sample_df['question'].notna()]
sample_df = sample_df[sample_df['question'].str.strip() != ""]

In [5]:
# BM25 to simulate retrieval
from rank_bm25 import BM25Okapi

# create a simple corpus
corpus = df['topic'].dropna().unique().tolist()

# tokenize
tokenized_corpus = [doc.lower().split() for doc in corpus]

# BM25 model
bm25 = BM25Okapi(tokenized_corpus)

def get_top_docs(query, k=3):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    
    top_n = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    
    docs = [corpus[i] for i in top_n]
    
    return docs if docs else ["Information is missing"]

In [6]:
# scale - apply to 30 quesries
sample_df['retrieved_docs'] = sample_df['topic'].apply(get_top_docs)

print(sample_df[['topic', 'retrieved_docs']].head(3))

                        topic  \
0     dutchess county tourism   
1  ontario california airport   
2        civil right movement   

                                      retrieved_docs  
0  [dutchess county tourism, cass county missouri...  
1  [ontario california airport, california franch...  
2  [civil right movement, battles in the civil wa...  


In [7]:
def generate_answer(query, docs):
    context = "\n".join([f"Doc{i+1}: {doc}" for i, doc in enumerate(docs[:2])])[:500]

    prompt = f"""
    Answer the question using the provided context.

    If the context is insufficient, say: Information is missing

    Keep the answer short.

    Question: {query}

    Context:
    {context}

    Answer:
    """

    response = ollama.chat(
        model='deepseek-llm:7b',
        messages=[{'role': 'user', 'content': prompt}],
        options={
            'num_predict': 80,
            'temperature': 0
        }
    )

    msg = response['message']
    answer = msg.get('content', '').strip()

    if not answer:
        answer = msg.get('thinking', '').strip()

    answer = answer.split("\n")[0].strip()

    # strict enforcement
    if "Information is missing" not in answer:
        # check if answer actually uses context keywords
        context_text = " ".join(docs[:2]).lower()
        overlap = any(word in context_text for word in answer.lower().split())

        if not overlap:
            return "Information is missing"

    return answer

In [8]:
# run on small sample
test_df = sample_df.head(3).copy()

answers = []
for _, row in test_df.iterrows():
    ans = generate_answer(row['topic'], row['retrieved_docs'])
    answers.append(ans)

test_df['answer'] = answers

In [9]:
pd.set_option('display.max_colwidth', None)
print(test_df[['topic', 'answer']])

                        topic  \
0     dutchess county tourism   
1  ontario california airport   
2        civil right movement   

                                                                                                                                                                                                                                                                                                                                       answer  
0                                                                                                                                                                             Dutchess County Tourism refers to a promotion of tourism within Dutchess County, New York. It does not provide specific information about Cass County Missouri.  
1                                                                                                                                                                                                  

In [40]:
# Baseline follow-up generation (plain prompting)
# This does NOT use answer or retrieved docs
# Represents "standard LLM behavior"

def generate_baseline_followup(query):
    prompt = f"""
    Generate ONE useful follow-up question to clarify the query.

    Rules:
    - Ask about a specific missing detail (e.g., location, type, features, time, etc.)
    - Do NOT ask vague questions like "What do you want to know?"
    - The question must narrow down the query and be relevant to the original query and potential information needs.
    - Frame the output as a question
    
    

    Query: {query}

    Return ONLY the question.
    """

    response = ollama.chat(
        model='deepseek-llm:7b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 50, 'temperature': 0}
    )

    msg = response['message']
    text = msg.get('content', '').strip()

    text = text.replace("Question:", "").replace("Follow-up", "").replace(":", "").strip()
    text = text.replace("One useful follow-up question to clarify the query would be,", "")
    text = text.replace("One useful follow-up question to clarify the query about", "")

    # remove placeholders
    text = text.replace("[location]", "")
    text = text.replace("[destination name]", "")
    text = text.replace("[activity/task]", "")
    text = text.replace("[insert diagnosis]", "")
    text = text.replace("[country/region]", "")

    # remove leftover bad patterns
    text = text.replace(" in ,", "")
    text = text.replace(" for ,", "")
    text = text.replace(" in ?", "?")
    text = text.replace(" for ?", "?")
    text = text.replace("  ", " ")

    text = text.strip()

    # keep only first question sentence
    if "?" in text:
        text = text.split("?")[0] + "?"

    return text

In [41]:
# test
print(generate_baseline_followup("flushing"))
print(generate_baseline_followup("ontario california airport"))

What specific type of flushing are you referring to, such as toilet or plumbing system?
What specific features or services are available at Ontario International Airport in California?


In [43]:
# Aspect extraction
def extract_aspects(query):
    prompt = f"""
    List 3 information aspects needed to answer the query.

    Return each aspect as a short phrase (2–5 words).
    Do not include numbering or explanations.

    Query: {query}

    Aspects:
    """

    response = ollama.chat(
        model='deepseek-llm:7b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 80, 'temperature': 0}
    )

    # cleaning function
    def clean_aspects(text):
        lines = text.split("\n")
        cleaned = []

        for line in lines:
            line = line.strip()

            # remove numbering / labels
            line = line.replace("Output", "")
            line = line.replace(":", "")
            line = line.lstrip("0123456789. ").strip()

            # keep only meaningful phrases
            if len(line.split()) >= 2:
                cleaned.append(line)

        return cleaned[:3]

    raw = response['message']['content']
    return clean_aspects(raw)

In [44]:
# test aspect extraction
print(extract_aspects("ontario california airport"))
print("------")
print(extract_aspects("flushing"))
print("------")
print(extract_aspects("civil right movement"))

['Airport name', 'Location details', 'Services and facilities']
------
['Definition/meaning of term', 'Context in which it is used', 'Related terms and concepts']
------
['Definition of Civil Rights Movement', 'Key Events and Figures in History', 'Impact on Society and Legislation']


In [45]:
def find_missing_aspect(query, answer, aspects):
    prompt = f"""
    Query: {query}

    Aspects:
    {aspects}

    Answer:
    {answer}

    Task:
    From the list of aspects above, select ONE aspect that is NOT covered in the answer.

    Rules:
    - You MUST choose ONLY from the provided aspects
    - Do NOT invent new aspects
    - Do NOT select aspects already clearly answered
    - Prefer the most important missing aspect
    - Output ONLY the exact aspect text
    - Match wording exactly as given in aspects

    Missing aspect:
    """

    response = ollama.chat(
        model='deepseek-llm:7b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 30, 'temperature': 0}
    )

    return response['message']['content'].strip()

In [46]:
# Improved follow-up generation
# Uses the generated answer to identify missing information
# This is the "hybrid approach"

def generate_aspect_followup(query, missing_aspect):
    prompt = f"""
    Generate ONE follow-up question to obtain this missing information:

    Missing aspect: {missing_aspect}

    Rules:
    - Ask a clear and specific question
    - Do NOT explain
    - Output ONLY the question
    """

    response = ollama.chat(
        model='deepseek-llm:7b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 50, 'temperature': 0}
    )

    text = response['message']['content'].strip()

    # remove placeholders
    text = text.replace("[location]", "")
    text = text.replace("[destination name]", "")
    text = text.replace("[activity/task]", "")
    text = text.replace("[insert diagnosis]", "")
    text = text.replace("[country/region]", "")

    # remove leftover bad patterns
    text = text.replace(" in ,", "")
    text = text.replace(" for ,", "")
    text = text.replace(" in ?", "?")
    text = text.replace(" for ?", "?")
    text = text.replace("  ", " ")

    text = text.strip()

    if "?" in text:
        text = text.split("?")[0] + "?"

    return text

In [47]:
# test improved follow-up question generation
query = "ontario california airport"
docs = get_top_docs(query)

ans = generate_answer(query, docs)
aspects = extract_aspects(query)
missing = find_missing_aspect(query, ans, aspects)
imp_q = generate_aspect_followup(query, missing)

print("Aspects:", aspects)
print("Answer:", ans)
print("Improved:", imp_q)


Aspects: ['Airport name', 'Location details', 'Services and facilities']
Answer: Ontario California Airport is a public use airport located in Ontario, California.
Improved: What services and facilities are available at the location?


In [49]:
def run_iterative_pipeline(query, steps=2):
    history = []
    context_query = query  # keep original

    for step in range(steps):
        docs = get_top_docs(context_query)
        answer = generate_answer(context_query, docs)

        aspects = extract_aspects(query)  # ALWAYS original query
        missing = find_missing_aspect(query, answer, aspects)
        followup = generate_aspect_followup(query, missing)

        history.append({
            "step": step + 1,
            "query": context_query,
            "answer": answer,
            "missing": missing,
            "followup": followup
        })

        # combine queries
        context_query = query + " " + followup

    return history

In [50]:
result = run_iterative_pipeline("ontario california airport")

for step in result:
    print(step)
    print("-----")

{'step': 1, 'query': 'ontario california airport', 'answer': 'Ontario California Airport is a public use airport located in Ontario, California.', 'missing': 'Services and facilities', 'followup': 'What services and facilities are available at the location?'}
-----
{'step': 2, 'query': 'ontario california airport What services and facilities are available at the location?', 'answer': 'Ontario, California has an airport called Ontario International Airport (ONT). At this airport, various services and facilities are available for passengers. These include restaurants, shops, currency exchange services, ATMs, charging stations for electronic devices, and restrooms. Additionally, the airport offers parking lots, rental car agencies, and ground transportation options such as taxis, shuttles, and limousines. For travelers', 'missing': 'Parking lots', 'followup': 'What type of parking facilities are available at the location?'}
-----


In [51]:
results = []

for query in sample_df['topic'].head(10):
    res = run_iterative_pipeline(query)
    results.append(res)

In [52]:
for i, res in enumerate(results):
    print(f"Query {i+1}")
    for step in res:
        print(step)
    print("------")

Query 1
{'step': 1, 'query': 'dutchess county tourism', 'answer': 'Dutchess County Tourism refers to a promotion of tourism within Dutchess County, New York. It does not provide specific information about Cass County Missouri.', 'missing': "'Tourism resources in Dutchess County'", 'followup': 'What types of tourism resources are available in Dutchess County, such as attractions, outdoor activities, or cultural sites?'}
{'step': 2, 'query': 'dutchess county tourism What types of tourism resources are available in Dutchess County, such as attractions, outdoor activities, or cultural sites?', 'answer': 'Dutchess County, New York offers a variety of tourism resources including attractions like the Vanderbilt Mansion National Historic Site and Blenheim Palace, outdoor activities such as hiking in the Ashokan Reservoir and horseback riding at Millbrook Riding Academy, and cultural sites like the Beekman 1802 Farmhouse and the Storm King Art Center.', 'missing': "'Accommodation options for to

In [53]:
final_rows = []

for _, row in sample_df.head(10).iterrows():
    query = row['topic']

    # baseline
    baseline_q = generate_baseline_followup(query)

    # proposed (single step)
    docs = get_top_docs(query)
    ans = generate_answer(query, docs)
    aspects = extract_aspects(query)
    missing = find_missing_aspect(query, ans, aspects)
    proposed_q = generate_aspect_followup(query, missing)

    # iterative
    iter_res = run_iterative_pipeline(query)

    iter1_q = iter_res[0]['followup']
    iter2_q = iter_res[1]['followup']

    final_rows.append({
        "Query": query,
        "Baseline": baseline_q,
        "Proposed": proposed_q,
        "Iter-1": iter1_q,
        "Iter-2": iter2_q
    })

In [54]:
final_df = pd.DataFrame(final_rows)
final_df.to_csv("final_results.csv", index=False)

#pd.set_option('display.max_colwidth', None)
#print(final_df)

In [55]:
test_df = df_small.copy()

In [ ]:
# Generate follow-ups for all queries
baseline_qs = []
proposed_qs = []

for _, row in test_df.iterrows():
    query = row['topic']

    # generate answer
    docs = get_top_docs(query)
    ans = generate_answer(query, docs)

    # baseline
    base_q = generate_baseline_followup(query)

    # proposed
    aspects = extract_aspects(query)
    missing = find_missing_aspect(query, ans, aspects)
    prop_q = generate_aspect_followup(query, missing)

    baseline_qs.append(base_q)
    proposed_qs.append(prop_q)


In [57]:
print(len(baseline_qs))
print(len(proposed_qs))

1240
1240


In [59]:
# take only the rows that were processed
partial_df = test_df.head(1240).copy()
partial_df['Baseline'] = baseline_qs
partial_df['Proposed'] = proposed_qs
partial_df.to_csv("checkpoint_1240.csv", index=False)
print("Saved!")

Saved!


In [60]:
print(partial_df['topic'].nunique())

88


In [61]:
df_eval = partial_df.drop_duplicates(subset=['topic'])

In [62]:
print(len(df_eval))

88


In [64]:
from bert_score import score

def compute_bertscore(preds, refs):
    P, R, F1 = score(preds, refs, lang="en", verbose=False)
    return F1.mean().item()

c:\Users\hardi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [65]:
refs = df_eval['question'].tolist()

baseline_score = compute_bertscore(df_eval['Baseline'].tolist(), refs)
proposed_score = compute_bertscore(df_eval['Proposed'].tolist(), refs)

print("Baseline:", baseline_score)
print("Proposed:", proposed_score)

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 1778.58it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 3344.20it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status    

Baseline: 0.8633370995521545
Proposed: 0.8510636687278748


In [66]:
# deduplicate the 88 completed topics
results_df = partial_df.drop_duplicates(subset=['topic']).copy()
print(len(results_df))  # should be 88

# compute BERTScore
refs = results_df['question'].tolist()
baseline_score = compute_bertscore(results_df['Baseline'].tolist(), refs)
proposed_score = compute_bertscore(results_df['Proposed'].tolist(), refs)

print("Baseline:", baseline_score)
print("Proposed:", proposed_score)

results_df.to_csv("results_88_unique.csv", index=False)

88


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2784.49it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2869.75it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status    

Baseline: 0.8633370995521545
Proposed: 0.8510636687278748


In [ ]:
# Store results
test_df['Baseline'] = baseline_qs
test_df['Proposed'] = proposed_qs

In [ ]:
# Prepare references
refs = test_df['question'].tolist()

In [ ]:
# BERTScore
from bert_score import score


def compute_bertscore(preds, refs):
    P, R, F1 = score(preds, refs, lang="en", verbose=False)
    return F1.mean().item()

In [ ]:
baseline_score = compute_bertscore(test_df['Baseline'].tolist(), refs)
proposed_score = compute_bertscore(test_df['Proposed'].tolist(), refs)

print("Baseline:", baseline_score)
print("Proposed:", proposed_score)